In [4]:
import pandas as pd
import sqlalchemy as db
from src.get_all_data import get_all_data
import joblib

In [2]:
engine = db.create_engine('sqlite:///../data/raw/data.db')
df = get_all_data(engine)

Searching for events

[+] US Federal Funds Rate
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Statement
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Press Conference
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Economic Projections
Skipped — next release on 2026-06-17 (not yet passed)

[+] US Core CPI m/m
Skipped — next release on 2026-04-10 (not yet passed)

[+] US CPI m/m
Skipped — next release on 2026-04-10 (not yet passed)

[+] US CPI y/y
Skipped — next release on 2026-04-10 (not yet passed)

[+] US PPI m/m
Skipped — next release on 2026-04-14 (not yet passed)

[+] US Core PCE Price Index m/m
Skipped — next release on 2026-04-09 (not yet passed)

[+] US Non-Farm Employment Change
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Unemployment Rate
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Average Hourly Earnings m/m
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Advance GD

In [5]:
hmm = joblib.load('../models/hmm_model.pkl')
pca = joblib.load('../models/pca.pkl')
scaler = joblib.load('../models/scaler.pkl')

X_scaled = scaler.transform(df)
X_pca    = pca.transform(X_scaled)
state_predict = hmm.predict(X_pca)

n_states = hmm.n_components

df['state'] = state_predict

In [7]:
df['spy_return'] = df['spy_close'].pct_change()
df['qqq_return'] = df['qqq_close'].pct_change()
df['^vix_return'] = df['^vix_close'].pct_change()
df['dx-y.nyb_return'] = df['dx-y.nyb_close'].pct_change()
df['gc=f_return'] = df['gc=f_close'].pct_change()

# ── REGIME RETURNS ─────
return_cols = {
    'spy_return':       'SPY',
    'qqq_return':       'QQQ',
    '^vix_return':      'VIX',
    'dx-y.nyb_return':  'DXY',
    'gc=f_return':      'Gold',
}

rows = []
for state in sorted(df['state'].unique()):
    mask = df['state'] == state
    row = {'state': state, 'count': mask.sum()}
    for col, name in return_cols.items():
        ret = df.loc[mask, col]
        row[f'{name}_mean']   = ret.mean()
        row[f'{name}_std']    = ret.std()
        row[f'{name}_sharpe'] = ret.mean() / ret.std()
    rows.append(row)

regime_profile = pd.DataFrame(rows).set_index('state')

#see all states
regime_profile

,count,SPY_mean,SPY_std,SPY_sharpe,QQQ_mean,QQQ_std,QQQ_sharpe,VIX_mean,VIX_std,VIX_sharpe,DXY_mean,DXY_std,DXY_sharpe,Gold_mean,Gold_std,Gold_sharpe
state,,,,,,,,,,,,,,,,
0,179,0.003404,0.020704,0.164411,0.004325,0.023161,0.186757,-0.003622,0.124353,-0.029128,-0.000036,0.006280,-0.005788,0.000073,0.018518,0.003922
1,428,-0.001474,0.025043,-0.058857,-0.001291,0.027516,-0.046905,0.021022,0.153211,0.137208,0.000082,0.008726,0.009438,0.002141,0.021334,0.100378
2,763,0.001126,0.013934,0.080799,0.001494,0.016223,0.092065,0.003950,0.093283,0.042346,0.000044,0.006921,0.006424,0.000594,0.017316,0.034288
3,601,0.001331,0.011546,0.115313,0.001824,0.014617,0.124793,0.003138,0.114783,0.027339,0.000258,0.006202,0.041585,0.000503,0.011989,0.041928
4,300,0.001866,0.010976,0.170021,0.002399,0.014827,0.161782,0.003517,0.099558,0.035323,0.000089,0.005560,0.016007,0.002194,0.011864,0.184940
